# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking2.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking2.parquet')

In [4]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,voting_lgbm_cat_xgb_hist_proba,stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.868510,0.826590
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.772514,0.801335
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.717012,0.782757
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,0.005494,0.502476
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,0.482416,0.703165


In [6]:
X_test.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,voting_lgbm_cat_xgb_hist_proba,stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,0.019849,0.509303
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,0.012173,0.505573
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,0.010304,0.504827
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.444558,0.696136
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.939648,0.852833


# Machine Learning

In [7]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "max_iter": trial.suggest_int("max_iter", 10, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.05, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 3),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 2, 8),
        "l2_regularization": trial.suggest_float("l2_regularization", 0.1, 100.0, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 50, 500),
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = HistGradientBoostingClassifier(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=500, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-18 15:55:01,051] A new study created in memory with name: no-name-094d8629-a8cb-4ad3-8149-0fae11d0eebb
Best trial: 10. Best value: 0.91625:   0%|          | 1/500 [00:07<1:05:29,  7.88s/it]

[I 2026-05-18 15:55:08,896] Trial 10 finished with value: 0.9162497731563265 and parameters: {'max_iter': 16, 'learning_rate': 0.023267473164318105, 'max_depth': 1, 'max_leaf_nodes': 8, 'l2_regularization': 34.294753403338376, 'min_samples_leaf': 367}. Best is trial 10 with value: 0.9162497731563265.


Best trial: 2. Best value: 0.934534:   0%|          | 2/500 [00:09<36:18,  4.38s/it]  

[I 2026-05-18 15:55:10,817] Trial 2 finished with value: 0.9345344306232238 and parameters: {'max_iter': 13, 'learning_rate': 0.002156777686842487, 'max_depth': 3, 'max_leaf_nodes': 5, 'l2_regularization': 0.6086362382567709, 'min_samples_leaf': 53}. Best is trial 2 with value: 0.9345344306232238.


Best trial: 0. Best value: 0.939494:   1%|          | 3/500 [00:10<21:55,  2.65s/it]

...

[I 2026-05-18 16:14:29,328] Trial 496 finished with value: 0.9496881636314093 and parameters: {'max_iter': 99, 'learning_rate': 0.03476794654505241, 'max_depth': 3, 'max_leaf_nodes': 8, 'l2_regularization': 0.962265477942625, 'min_samples_leaf': 336}. Best is trial 188 with value: 0.9497571771545005.
[I 2026-05-18 16:14:29,431] Trial 497 finished with value: 0.9496916741164512 and parameters: {'max_iter': 99, 'learning_rate': 0.035967002811773174, 'max_depth': 3, 'max_leaf_nodes': 8, 'l2_regularization': 0.9584148324083663, 'min_samples_leaf': 344}. Best is trial 188 with value: 0.9497571771545005.


Best trial: 188. Best value: 0.949757: 100%|█████████▉| 499/500 [19:29<00:00,  1.10it/s]

[I 2026-05-18 16:14:30,425] Trial 498 finished with value: 0.9496899307081058 and parameters: {'max_iter': 99, 'learning_rate': 0.0406260443954582, 'max_depth': 3, 'max_leaf_nodes': 8, 'l2_regularization': 1.6974328330950201, 'min_samples_leaf': 67}. Best is trial 188 with value: 0.9497571771545005.


Best trial: 188. Best value: 0.949757: 100%|██████████| 500/500 [19:29<00:00,  2.34s/it]

[I 2026-05-18 16:14:30,917] Trial 499 finished with value: 0.9497146325270144 and parameters: {'max_iter': 100, 'learning_rate': 0.043586054597267186, 'max_depth': 3, 'max_leaf_nodes': 8, 'l2_regularization': 0.9412982490993115, 'min_samples_leaf': 330}. Best is trial 188 with value: 0.9497571771545005.
Best trial score:
0.9497571771545005

Best params:
{'max_iter': 100, 'learning_rate': 0.049279424968777505, 'max_depth': 3, 'max_leaf_nodes': 8, 'l2_regularization': 0.7407240315476109, 'min_samples_leaf': 65}


In [8]:
stacking_lgbm = HistGradientBoostingClassifier(**study.best_params).fit(X_train, y_train.PitNextLap)

# Submission

In [9]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [10]:
submission['PitNextLap'] = stacking_lgbm.predict_proba(X_test)[:, 1]

In [11]:
submission.to_csv('../data/submission/submission.csv', index=False)

In [12]:
X_train.columns

Index(['lgbm_proba', 'cat_proba', 'xgb_proba', 'hist_proba', 'lg_proba',
       'knn_proba', 'voting_lgbm_cat_xgb_hist_proba',
       'stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba'],
      dtype='str')